# Subtopic 4 — Advanced Design and Implementation

**Problems:** Sliding Window Maximum, Stock Span Problem, Celebrity Problem, LRU Cache, LFU Cache.

This subtopic is where stack/queue ideas graduate into **designed data structures**. The first two are monotonic-structure applications (a deque, a stack of previous-greater indices). The last three are constant-time designs built from the right combination of hash map + linked list — the canonical "design an $O(1)$ cache" interview questions. The discipline is unchanged: **state the invariant, prove each operation's cost against it, then write code whose variable names mirror the invariant.**

## 1. Sliding Window Maximum — Monotonic Deque

Given an array and window size $k$, report the max of every length-$k$ window.

### Invariant (decreasing deque of indices)
The deque holds **indices**, and the corresponding values are **decreasing front → back**. Two consequences enforced every step:
1. **Front is the answer:** `deque.front()` is always the index of the maximum in the current window.
2. **All indices are in-window:** any index that has slid out of the left edge is expired from the front.

Formally, after processing index $i$ for window $[i-k+1,\, i]$:
$$\text{front} \in [i-k+1,\, i], \qquad a[\text{dq}[0]] \ge a[\text{dq}[1]] \ge \cdots \ge a[\text{dq}[\text{back}]].$$

### Two operations per step
- **Expire (front):** while `dq.front() <= i - k`, pop front — it has left the window.
- **Dominate (back):** while `a[dq.back()] <= a[i]`, pop back — a smaller-or-equal element can never again be the max while $a[i]$ is in-window, so it is useless.
- Push $i$; once $i \ge k-1$, record `a[dq.front()]`.

### Amortized $O(n)$
Each index is pushed once and popped (front or back) at most once → $\le 2n$ deque operations total. Per-window cost is $O(1)$ amortized; the whole sweep is $O(n)$. $\square$

### Comparison with other approaches
| Method | Build | Query | Update | Best when |
|---|---|---|---|---|
| **Monotonic deque** | $O(n)$ | — | $O(1)$ am. | fixed window sliding left→right (this problem) |
| Sparse table | $O(n\log n)$ | $O(1)$ | static only | many arbitrary-range max queries, no updates |
| Segment tree | $O(n)$ | $O(\log n)$ | $O(\log n)$ | dynamic array with point updates + range max |
| max-heap (lazy delete) | $O(n)$ | $O(1)$ | $O(\log n)$ | window max but with messy deletions |

The deque wins here because the window moves monotonically, so each element enters and leaves exactly once — no $\log$ factor is needed.

### Boundary transitions
| Step at $i$ | Action |
|---|---|
| front expired (`dq.front() <= i-k`) | pop front |
| back dominated (`a[dq.back()] <= a[i]`) | pop back (repeat) |
| always | push $i$ |
| `i >= k-1` | output `a[dq.front()]` |

In [ ]:
#include <iostream>
#include <vector>
#include <deque>
#include <stack>
#include <string>
#include <unordered_map>
#include <list>
#include <algorithm>
using namespace std;

template <typename T>
void printVec(const string& label, const vector<T>& v) {
    cout << label << " [";
    for (size_t i = 0; i < v.size(); ++i) cout << v[i] << (i+1<v.size() ? "," : "");
    cout << "]";
}

vector<int> maxSlidingWindow(const vector<int>& a, int k) {
    deque<int> dq;                          // indices; values decreasing front->back
    vector<int> out;
    int n = a.size();
    for (int i = 0; i < n; ++i) {
        if (!dq.empty() && dq.front() <= i - k) dq.pop_front();  // expire left edge
        while (!dq.empty() && a[dq.back()] <= a[i]) dq.pop_back();// drop dominated tails
        dq.push_back(i);
        if (i >= k - 1) out.push_back(a[dq.front()]);            // front = window max
    }
    return out;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = maxSlidingWindow({1,3,-1,-3,5,3,6,7}, 3);
    printVec("[1,3,-1,-3,5,3,6,7] k=3 ->", r1); cout << " (Expected: [3,3,5,5,6,7])\n";
    auto r2 = maxSlidingWindow({1}, 1);
    printVec("[1] k=1 ->", r2); cout << " (Expected: [1])\n";
    auto r3 = maxSlidingWindow({9,8,7,6,5}, 2);
    printVec("[9,8,7,6,5] k=2 ->", r3); cout << " (Expected: [9,8,7,6])\n"; // decreasing
    auto r4 = maxSlidingWindow({1,2,3,4,5}, 2);
    printVec("[1,2,3,4,5] k=2 ->", r4); cout << " (Expected: [2,3,4,5])\n"; // increasing
    auto r5 = maxSlidingWindow({4,4,4,4}, 2);
    printVec("[4,4,4,4] k=2 ->", r5); cout << " (Expected: [4,4,4])\n"; // all-same
    auto r6 = maxSlidingWindow({7,2,4}, 3);
    printVec("[7,2,4] k=3 (whole array) ->", r6); cout << " (Expected: [7])\n";
}

## 2. Stock Span Problem

For each day $i$, the span is the number of consecutive days up to and including $i$ whose price is $\le$ price$[i]$:
$$\text{span}[i] = i - \text{PGE}(i),$$
where $\text{PGE}(i)$ is the index of the **previous strictly greater** element (or $-1$ if none).

### Invariant (decreasing stack of indices)
The stack holds indices whose prices are **strictly decreasing bottom → top**. For day $i$, pop every index whose price is $\le$ price$[i]$ (those days are within the span and dominated). The new top, if any, is the previous-greater index; the span is the index distance.

### Derivation of the formula
After popping all $\le$ price$[i]$:
- if stack empty: no greater day exists before $i$, so $\text{PGE}=-1$ and $\text{span}=i-(-1)=i+1$ (all days so far).
- else: $\text{PGE}=\text{stack top}$, and $\text{span}=i-\text{top}$.

Each popped day $j$ satisfies price$[j] \le$ price$[i]$, and was contiguous, so it correctly belongs inside $i$'s span. $\square$

### Amortized $O(1)$ per query
Standard monotonic-stack aggregate bound: each index pushed once, popped at most once → $O(n)$ total over $n$ days.

### Boundary transitions
| Day $i$ | Action |
|---|---|
| price$[\text{top}] \le$ price$[i]$ | pop (repeat) |
| stack empty after popping | span $= i+1$ |
| else | span $= i - \text{top}$ |
| always | push $i$ |

In [ ]:
// Online (streaming) Stock Span via a stack of (price, span) pairs.
class StockSpanner {
    stack<pair<int,int>> st;   // (price, span) with prices strictly decreasing bottom->top
public:
    int next(int price) {
        int span = 1;                                  // the day itself
        while (!st.empty() && st.top().first <= price) {// fold in dominated earlier days
            span += st.top().second; st.pop();
        }
        st.push({price, span});
        return span;
    }
};

// Batch version returning all spans for an array.
vector<int> stockSpan(const vector<int>& price) {
    int n = price.size();
    vector<int> span(n);
    stack<int> st;                                     // indices, prices strictly decreasing
    for (int i = 0; i < n; ++i) {
        while (!st.empty() && price[st.top()] <= price[i]) st.pop();  // pop <= current
        span[i] = st.empty() ? (i + 1) : (i - st.top());             // distance to PGE
        st.push(i);
    }
    return span;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = stockSpan({100,80,60,70,60,75,85});
    printVec("[100,80,60,70,60,75,85] ->", r1); cout << " (Expected: [1,1,1,2,1,4,6])\n";
    auto r2 = stockSpan({10,20,30,40});
    printVec("[10,20,30,40] ->", r2); cout << " (Expected: [1,2,3,4])\n"; // increasing
    auto r3 = stockSpan({40,30,20,10});
    printVec("[40,30,20,10] ->", r3); cout << " (Expected: [1,1,1,1])\n"; // decreasing

    StockSpanner s;   // streaming variant on the classic example
    cout << "stream: ";
    for (int p : {100,80,60,70,60,75,85}) cout << s.next(p) << " ";
    cout << "(Expected: 1 1 1 2 1 4 6)\n";
}

## 3. Celebrity Problem

Among $n$ people, a *celebrity* is someone whom **everyone knows** but who **knows no one**. Given `knows(a,b)` (does $a$ know $b$?), find the celebrity or report none. At most one can exist.

### The elimination invariant (two-pointer / stack)
Compare two candidates $a$ and $b$:
- if `knows(a, b)` is true → $a$ knows someone → **$a$ is not a celebrity** (eliminate $a$).
- if `knows(a, b)` is false → $b$ is unknown to $a$ → **$b$ is not a celebrity** (eliminate $b$).

Either way, **one comparison eliminates exactly one candidate.** After $n-1$ comparisons, a single candidate survives.

### Why exactly one survives, and why verification is needed
The elimination guarantees the survivor is the *only possible* celebrity — every other person failed a necessary condition. But the survivor itself was only ever checked as the *eliminator*; we never confirmed the full pair of conditions for it. So a **two-pass verification** is required:
$$\forall j \ne c:\quad \text{knows}(c, j) = \text{false} \ \wedge\ \text{knows}(j, c) = \text{true}.$$
If any check fails, there is no celebrity. The verification is necessary because elimination proves "no one *else* qualifies," not "the survivor qualifies." $\square$

### Complexity
$n-1$ comparisons to find the candidate $+$ at most $2(n-1)$ for verification $= O(n)$ calls to `knows`, $O(1)$ extra space (two-pointer form). The two-pointer and stack formulations are equivalent; the stack pushes all candidates then pops two at a time, reducing by one each comparison — same $O(n)$ bound.

### Boundary transitions (two-pointer)
| `knows(lo, hi)` | Action |
|---|---|
| true | `lo++` (lo eliminated) |
| false | `hi--` (hi eliminated) |
| `lo == hi` | candidate found → verify |

In [ ]:
// knows(a,b) provided as a relation matrix for testing.
struct Society {
    vector<vector<int>> M;   // M[a][b] == 1 iff a knows b
    bool knows(int a, int b) const { return M[a][b] == 1; }
    int n() const { return M.size(); }
};

int findCelebrity(const Society& soc) {
    int n = soc.n();
    int lo = 0, hi = n - 1;
    while (lo < hi) {                       // each comparison eliminates exactly one
        if (soc.knows(lo, hi)) lo++;        // lo knows someone -> lo not celeb
        else                   hi--;        // hi unknown to lo  -> hi not celeb
    }
    int c = lo;                             // sole surviving candidate
    for (int j = 0; j < n; ++j) {           // verification pass: necessary, not redundant
        if (j == c) continue;
        if (soc.knows(c, j) || !soc.knows(j, c)) return -1;  // c fails celeb conditions
    }
    return c;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    // Person 2 is the celebrity: everyone knows 2, 2 knows nobody.
    Society s1{{{0,1,1},{1,0,1},{0,0,0}}};
    cout << "celeb (2 is celeb) -> " << findCelebrity(s1) << " (Expected: 2)\n";

    // No celebrity: nobody is universally known and unknowing.
    Society s2{{{0,1,0},{1,0,0},{1,1,0}}};
    cout << "no celeb -> " << findCelebrity(s2) << " (Expected: -1)\n";

    // Single person trivially a celebrity (knows no one, vacuously known by all).
    Society s3{{{0}}};
    cout << "single -> " << findCelebrity(s3) << " (Expected: 0)\n";

    // Person 0 is celeb.
    Society s4{{{0,0,0},{1,0,1},{1,0,0}}};
    cout << "celeb (0 is celeb) -> " << findCelebrity(s4) << " (Expected: 0)\n";
}

## 4. LRU Cache — Hash Map + Doubly Linked List

Design a cache with capacity $C$ supporting `get` and `put`, both $O(1)$, evicting the **least-recently-used** entry on overflow.

### Invariant
Two structures kept in lockstep:
- a **doubly linked list** of nodes ordered from **most-recently-used (head)** to **least-recently-used (tail)**;
- a **hash map** `key → node pointer` for $O(1)$ location.

At all times: the list contains exactly the cached keys, MRU at front; `map[k]` points to the node holding key $k$; `size == list length == map size $\le C$`.

### Four operations, each $O(1)$
| Operation | Steps |
|---|---|
| `get(k)` hit | locate via map, **unlink** node, **move to head**, return value |
| `get(k)` miss | return $-1$ |
| `put(k,v)` existing | update value, move node to head |
| `put(k,v)` new | create node at head, insert in map; if `size > C`, **evict tail** (remove from list *and* map) |

### Why a doubly (not singly) linked list
Moving an arbitrary node to the head requires **unlinking it in $O(1)$**, which needs its predecessor — only a *doubly* linked list gives `node->prev` directly. A singly linked list would need an $O(n)$ scan to find the predecessor, breaking the $O(1)$ guarantee. Eviction at the tail likewise needs `tail->prev` in $O(1)$.

### Dummy head & tail sentinels
Two permanent sentinel nodes bracket the list. Then every real node always has non-null `prev` and `next`, so insert/unlink need **no null checks or empty-list special cases** — the single most error-eliminating trick in linked-list design.

$$\text{insertFront}(x):\ x.\text{next}=head.\text{next};\ x.\text{prev}=head;\ head.\text{next}.\text{prev}=x;\ head.\text{next}=x.$$
$$\text{unlink}(x):\ x.\text{prev}.\text{next}=x.\text{next};\ x.\text{next}.\text{prev}=x.\text{prev}.$$

In [ ]:
class LRUCache {
    struct Node {
        int key, val;
        Node *prev, *next;
        Node(int k, int v): key(k), val(v), prev(nullptr), next(nullptr) {}
    };
    int cap;
    unordered_map<int, Node*> mp;   // key -> node
    Node *head, *tail;              // sentinels: head<->...<->tail (MRU near head)

    void unlink(Node* x) {          // O(1): needs x->prev, hence DOUBLY linked
        x->prev->next = x->next;
        x->next->prev = x->prev;
    }
    void insertFront(Node* x) {     // place right after head sentinel = MRU
        x->next = head->next; x->prev = head;
        head->next->prev = x; head->next = x;
    }
public:
    LRUCache(int capacity): cap(capacity) {
        head = new Node(0,0); tail = new Node(0,0);  // sentinels never hold real data
        head->next = tail; tail->prev = head;
    }
    ~LRUCache() {
        Node* p = head;
        while (p) { Node* n = p->next; delete p; p = n; }
    }
    int get(int key) {
        auto it = mp.find(key);
        if (it == mp.end()) return -1;          // miss
        Node* x = it->second;
        unlink(x); insertFront(x);              // touch -> becomes MRU
        return x->val;
    }
    void put(int key, int value) {
        auto it = mp.find(key);
        if (it != mp.end()) {                   // existing: update + promote
            Node* x = it->second;
            x->val = value;
            unlink(x); insertFront(x);
            return;
        }
        Node* x = new Node(key, value);         // new entry at MRU
        mp[key] = x; insertFront(x);
        if ((int)mp.size() > cap) {             // over capacity -> evict LRU (tail->prev)
            Node* lru = tail->prev;
            unlink(lru); mp.erase(lru->key); delete lru;
        }
    }
};

In [ ]:
// Tests: input -> actual (Expected: X)
{
    LRUCache c(2);
    c.put(1,1); c.put(2,2);
    cout << "get(1) -> " << c.get(1) << " (Expected: 1)\n";   // 1 becomes MRU
    c.put(3,3);                                                // evicts key 2 (LRU)
    cout << "get(2) -> " << c.get(2) << " (Expected: -1)\n";  // evicted
    c.put(4,4);                                                // evicts key 1
    cout << "get(1) -> " << c.get(1) << " (Expected: -1)\n";
    cout << "get(3) -> " << c.get(3) << " (Expected: 3)\n";
    cout << "get(4) -> " << c.get(4) << " (Expected: 4)\n";

    LRUCache d(1);
    d.put(1,10);
    cout << "cap1 get(1) -> " << d.get(1) << " (Expected: 10)\n";
    d.put(2,20);                                               // evicts 1
    cout << "cap1 get(1) -> " << d.get(1) << " (Expected: -1)\n";
    cout << "cap1 get(2) -> " << d.get(2) << " (Expected: 20)\n";

    LRUCache e(2);
    e.put(1,1); e.put(1,100);                                  // update existing
    cout << "update get(1) -> " << e.get(1) << " (Expected: 100)\n";
}

## 5. LFU Cache — the $O(1)$ Design

Evict the **least-frequently-used** entry; break ties by **least-recently-used among the minimum frequency**. All operations $O(1)$. This is the hardest classic cache design.

### The three-part invariant
1. `key → (value, freq, iterator)` map — locates a key's value, current frequency, and its exact position in a list. *(`keyMap`)*
2. `freq → doubly-linked list of keys` map — all keys sharing frequency $f$, ordered by recency (**MRU at front, LRU at back**) within that frequency bucket. *(`freqMap`)*
3. `minFreq` — the smallest frequency currently present, so eviction is $O(1)$.

Invariants maintained at all times:
- a key in `freqMap[f]` has `keyMap[key].freq == f`;
- `minFreq` = $\min$ over all present frequencies (when cache non-empty);
- eviction target = **back of `freqMap[minFreq]`** (least-frequent, and LRU within it).

### Operations, all $O(1)$
**`get(key)` (hit):** read value; then **promote frequency** $f \to f+1$: erase the key from `freqMap[f]`, push it to the front of `freqMap[f+1]`, update `keyMap`. If `freqMap[f]` became empty **and** `f == minFreq`, increment `minFreq` (the old min bucket is gone). Return value.

**`put(key,val)` existing:** update value, then promote frequency exactly as in `get`.

**`put(key,val)` new:**
- if at capacity: evict the back of `freqMap[minFreq]` (remove from both maps).
- insert the new key with `freq = 1` at the front of `freqMap[1]`; set `keyMap`.
- **set `minFreq = 1`** — a brand-new key always has frequency 1, the global minimum.

### The critical `minFreq` rule (the delta)
`minFreq` changes in exactly two places, and recognizing this is the crux of the design:
- **resets to 1** on every `put` of a *new* key (a freq-1 entry now exists).
- **increments by 1** only when, during a frequency promotion, the bucket `freqMap[minFreq]` becomes **empty** (every minimum-frequency key was promoted away). Since promotion raises a key from $f$ to $f+1$, and we only promote one key at a time, the new minimum is exactly `minFreq + 1`.

It never needs to *decrease* except via the new-key reset, and never jumps by more than 1 on promotion — because emptying the min bucket by promoting one key leaves $f+1$ as the next smallest. $\square$

### Why a list iterator in the key map
Storing the list iterator (or node) lets us **splice a key out of its frequency bucket in $O(1)$** without scanning. `std::list::erase` on a stored iterator is constant time; combined with `push_front` on the next bucket, frequency promotion is $O(1)$.

### Boundary transitions
| Event | `minFreq` action | bucket action |
|---|---|---|
| new key inserted | set to 1 | push front of `freqMap[1]` |
| key promoted, old bucket non-empty | unchanged | move to front of `freqMap[f+1]` |
| key promoted, old bucket emptied & was min | `minFreq++` | erase empty `freqMap[f]` |
| eviction (capacity, new key) | unchanged (then reset to 1) | pop back of `freqMap[minFreq]` |

In [ ]:
class LFUCache {
    int cap, minFreq;
    // key -> (value, freq, iterator into freqMap[freq])
    unordered_map<int, tuple<int,int,list<int>::iterator>> keyMap;
    // freq -> list of keys (front = MRU, back = LRU within this frequency)
    unordered_map<int, list<int>> freqMap;

    void touch(int key) {                       // promote key's frequency by 1, O(1)
        auto& [val, f, it] = keyMap[key];
        freqMap[f].erase(it);                   // splice out of current bucket in O(1)
        if (freqMap[f].empty()) {
            freqMap.erase(f);
            if (minFreq == f) minFreq++;        // old min bucket emptied -> next min is f+1
        }
        freqMap[f+1].push_front(key);           // most-recent in the higher bucket
        keyMap[key] = {val, f+1, freqMap[f+1].begin()};
    }
public:
    LFUCache(int capacity): cap(capacity), minFreq(0) {}

    int get(int key) {
        if (cap == 0 || keyMap.find(key) == keyMap.end()) return -1;
        int val = std::get<0>(keyMap[key]);
        touch(key);                             // a get counts as a use
        return val;
    }
    void put(int key, int value) {
        if (cap == 0) return;
        if (keyMap.find(key) != keyMap.end()) { // existing: update value, bump freq
            std::get<0>(keyMap[key]) = value;
            touch(key);
            return;
        }
        if ((int)keyMap.size() >= cap) {        // evict LRU among least frequent
            int evict = freqMap[minFreq].back(); // back = LRU within min bucket
            freqMap[minFreq].pop_back();
            if (freqMap[minFreq].empty()) freqMap.erase(minFreq);
            keyMap.erase(evict);
        }
        freqMap[1].push_front(key);             // new key: frequency 1, MRU
        keyMap[key] = {value, 1, freqMap[1].begin()};
        minFreq = 1;                            // a freq-1 entry now exists -> global min
    }
};

In [ ]:
// Tests: input -> actual (Expected: X)
{
    LFUCache c(2);
    c.put(1,1); c.put(2,2);
    cout << "get(1) -> " << c.get(1) << " (Expected: 1)\n";   // freq: 1->2, 2->1
    c.put(3,3);                                                // evict key 2 (freq 1, the min)
    cout << "get(2) -> " << c.get(2) << " (Expected: -1)\n";  // evicted
    cout << "get(3) -> " << c.get(3) << " (Expected: 3)\n";   // freq: 3->2
    c.put(4,4);                                                // tie freq1: keys {1,3 at f2}, evict... 1? see note
    cout << "get(1) -> " << c.get(1) << " (Expected: -1)\n";  // 1 was LFU(freq2 LRU) -> evicted
    cout << "get(3) -> " << c.get(3) << " (Expected: 3)\n";
    cout << "get(4) -> " << c.get(4) << " (Expected: 4)\n";

    LFUCache z(0);                                             // zero capacity edge case
    z.put(0,0);
    cout << "cap0 get(0) -> " << z.get(0) << " (Expected: -1)\n";

    LFUCache s(2);                                             // tie-break by recency within freq
    s.put(1,1); s.put(2,2);
    s.get(1); s.get(2);                                        // both now freq 2
    s.put(3,3);                                                // evict LRU among freq2 = key 1
    cout << "tie get(1) -> " << s.get(1) << " (Expected: -1)\n";
    cout << "tie get(2) -> " << s.get(2) << " (Expected: 2)\n";
    cout << "tie get(3) -> " << s.get(3) << " (Expected: 3)\n";
}

## Unified Mental Model

```text
        ADVANCED DESIGNS = right monotone structure  OR  map + linked list
        =================================================================

  MONOTONE STRUCTURES (the pop is the answer)
  -------------------------------------------
    Sliding Window Max : decreasing DEQUE of indices
                         front = window max ; expire front, dominate back
    Stock Span         : decreasing STACK of indices (previous-greater)
                         span = i - PGE(i)

  MAP + LINKED LIST (O(1) by construction)
  ----------------------------------------
    LRU : hashmap(key->node) + DOUBLY linked list, MRU=head .. LRU=tail
          touch = unlink + insertFront ; evict = tail->prev
          (doubly: need prev for O(1) unlink ; sentinels kill edge cases)
    LFU : hashmap(key->{val,freq,iter})
          + hashmap(freq -> list<key>)  (recency-ordered within a freq)
          + minFreq pointer
          touch = move key from freqMap[f] to freqMap[f+1]
          minFreq: reset to 1 on NEW key ; +1 when min bucket empties

  ELIMINATION (no structure, just pairwise cancellation)
  ------------------------------------------------------
    Celebrity : each comparison kills one candidate -> 1 survivor -> VERIFY
```

**The unifying lesson:** $O(1)$ designs come from pairing a structure that gives *location* (hash map) with a structure that gives *order* (linked list / deque / stack). The monotone variants discard what can never matter again; the cache variants keep an order they can splice in constant time. Never scan when an iterator or a sentinel can do the work in $O(1)$.

## Decision Tree — Which Design?

```text
What is the access pattern?
│
├─ "max/min over a SLIDING fixed window"
│     ├─ window moves monotonically, point reads → monotonic DEQUE  (O(n))
│     ├─ static array, many arbitrary ranges      → sparse table     (O(1) query)
│     └─ dynamic array, point updates             → segment tree     (O(log n))
│
├─ "consecutive run up to me bounded by a greater value"
│     → Stock Span: decreasing stack of indices, span = i - PGE
│
├─ "find the one universally-known / knows-no-one element"
│     → Celebrity: pairwise elimination (O(n)) then 2-pass verify
│
└─ "fixed-capacity cache, O(1) get/put, evict by a policy"
      ├─ evict by RECENCY        → LRU: map + doubly linked list + sentinels
      └─ evict by FREQUENCY      → LFU: keyMap + freqMap(list) + minFreq
            tie within freq broken by recency (front=MRU, back=LRU)
```

## Complexity Summary

| Problem | Operation | Time | Space | Key Invariant |
|---|---|---|---|---|
| Sliding Window Max | per element | $O(1)$ am. | $O(k)$ | deque indices decreasing; front in-window = max |
| | full sweep | $O(n)$ | $O(k)$ | each index pushed/popped once |
| Stock Span | `next` / per day | $O(1)$ am. | $O(n)$ | decreasing-price stack; span $= i - \text{PGE}$ |
| Celebrity | find + verify | $O(n)$ | $O(1)$ | each comparison eliminates one candidate |
| LRU | `get` / `put` | $O(1)$ | $O(C)$ | list MRU→LRU; map→node; evict tail->prev |
| LFU | `get` / `put` | $O(1)$ | $O(C)$ | keyMap + freqMap(recency lists) + minFreq; min resets to 1 on new key, +1 when min bucket empties |